# Notebook 04 — Modelo de Predição de Voto

**Sprint 3 — Lei e Política**

Modelo supervisionado para testar a hipótese central do projeto: **prever o voto
individual de um deputado (favorável/contrário) com acurácia ≥ 70% em split temporal**.

Escopo: **Câmara** (só há votos nominais da Câmara; o Senado coletou apenas matérias).

## Pipeline

1. Montagem do dataset voto-a-voto (junção `votos`×`votacoes`×`proposicoes`×`parlamentares`)
2. Split temporal 80/20 por data da votação (sem vazamento)
3. Features: `partido`, `uf`, `tema_cluster`, `mes` + tendência partido×tema (derivada só do treino)
4. RandomForestClassifier — treino e avaliação (acurácia, F1 macro, matriz de confusão, importâncias)
5. Gravação dos resultados em `metricas_modelo`

In [1]:
import sys
sys.path.insert(0, '..')

import logging

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    confusion_matrix,
    classification_report,
)

from src.db import buscar_todos, inserir_metricas

logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s')
log = logging.getLogger('04_modelo')
print('Módulos carregados.')

Módulos carregados.


## 1. Montagem do dataset voto-a-voto

Cada linha é um voto nominal de um deputado em uma votação. Juntamos `votos` →
`votacoes` (data, proposição) → `proposicoes` (tema_cluster) → `parlamentares`
(partido, uf). Mantemos apenas votos decisivos (`favoravel`/`contrario`).

In [2]:
# Carrega as 4 tabelas (escopo: Câmara — só há votos nominais da Câmara)
votos = pd.DataFrame(buscar_todos('votos', 'votacao_id,parlamentar_id,voto'))
votacoes = pd.DataFrame(buscar_todos('votacoes', 'id,proposicao_id,data'))
proposicoes = pd.DataFrame(buscar_todos('proposicoes', 'id,tema_cluster'))
parlamentares = pd.DataFrame(buscar_todos('parlamentares', 'id,partido,uf,casa'))

print(f'votos={len(votos)}  votacoes={len(votacoes)}  '
      f'proposicoes={len(proposicoes)}  parlamentares={len(parlamentares)}')

# Chaves de junção como inteiro anulável (proposicao_id vem como object por conter nulos)
for d, col in [(votos, 'votacao_id'), (votos, 'parlamentar_id'),
               (votacoes, 'id'), (votacoes, 'proposicao_id'),
               (proposicoes, 'id'), (parlamentares, 'id')]:
    d[col] = pd.to_numeric(d[col], errors='coerce').astype('Int64')

# Junções voto-a-voto
df = votos.merge(votacoes, left_on='votacao_id', right_on='id', suffixes=('', '_vt'))
df = df.merge(proposicoes, left_on='proposicao_id', right_on='id',
              how='left', suffixes=('', '_pr'))
df = df.merge(parlamentares, left_on='parlamentar_id', right_on='id',
              how='left', suffixes=('', '_pl'))

# Apenas Câmara e voto binário decisivo (abstenções fora — hipótese é binária)
df = df[df['casa'] == 'camara']
df = df[df['voto'].isin(['favoravel', 'contrario'])].copy()

# tema_cluster ausente (votação sem proposição linkada) -> categoria 'sem_tema' (-1)
df['tema_cluster'] = df['tema_cluster'].fillna(-1).astype(int)

# data -> datetime; descarta linhas sem data (necessárias para o split temporal)
df['data'] = pd.to_datetime(df['data'], errors='coerce')
df = df[df['data'].notna()].copy()

# preenche partido/uf ausentes para não perder linhas no one-hot
df['partido'] = df['partido'].fillna('DESCONHECIDO')
df['uf'] = df['uf'].fillna('XX')

# alvo binário: 1 = favoravel, 0 = contrario
df['alvo'] = (df['voto'] == 'favoravel').astype(int)

print(f'\nDataset final: {len(df)} votos')
print('Distribuição de classes:')
print(df['voto'].value_counts())

2026-06-25 23:57:10,730 [INFO] HTTP Request: GET https://woxluznsfqskopzzjqtf.supabase.co/rest/v1/votos?select=votacao_id%2Cparlamentar_id%2Cvoto&offset=0&limit=1000 "HTTP/2 200 OK"


2026-06-25 23:57:10,981 [INFO] HTTP Request: GET https://woxluznsfqskopzzjqtf.supabase.co/rest/v1/votos?select=votacao_id%2Cparlamentar_id%2Cvoto&offset=1000&limit=1000 "HTTP/2 200 OK"


2026-06-25 23:57:11,281 [INFO] HTTP Request: GET https://woxluznsfqskopzzjqtf.supabase.co/rest/v1/votos?select=votacao_id%2Cparlamentar_id%2Cvoto&offset=2000&limit=1000 "HTTP/2 200 OK"


2026-06-25 23:57:11,565 [INFO] HTTP Request: GET https://woxluznsfqskopzzjqtf.supabase.co/rest/v1/votos?select=votacao_id%2Cparlamentar_id%2Cvoto&offset=3000&limit=1000 "HTTP/2 200 OK"


2026-06-25 23:57:11,870 [INFO] HTTP Request: GET https://woxluznsfqskopzzjqtf.supabase.co/rest/v1/votos?select=votacao_id%2Cparlamentar_id%2Cvoto&offset=4000&limit=1000 "HTTP/2 200 OK"


...

2026-06-25 23:57:36,402 [INFO] HTTP Request: GET https://woxluznsfqskopzzjqtf.supabase.co/rest/v1/proposicoes?select=id%2Ctema_cluster&offset=20000&limit=1000 "HTTP/2 200 OK"


2026-06-25 23:57:36,600 [INFO] HTTP Request: GET https://woxluznsfqskopzzjqtf.supabase.co/rest/v1/proposicoes?select=id%2Ctema_cluster&offset=21000&limit=1000 "HTTP/2 200 OK"


2026-06-25 23:57:36,912 [INFO] HTTP Request: GET https://woxluznsfqskopzzjqtf.supabase.co/rest/v1/proposicoes?select=id%2Ctema_cluster&offset=22000&limit=1000 "HTTP/2 200 OK"


2026-06-25 23:57:37,138 [INFO] HTTP Request: GET https://woxluznsfqskopzzjqtf.supabase.co/rest/v1/parlamentares?select=id%2Cpartido%2Cuf%2Ccasa&offset=0&limit=1000 "HTTP/2 200 OK"


votos=55570  votacoes=137  proposicoes=22106  parlamentares=726

Dataset final: 55233 votos
Distribuição de classes:
voto
favoravel    35206
contrario    20027
Name: count, dtype: int64


## 2. Split temporal

Ordenamos os votos pela data da votação e cortamos no **percentil 80** das datas:
treino = votações mais antigas, teste = mais recentes. Isso simula prever votos futuros a
partir do histórico — o split aleatório seria irrealista e otimista.

In [3]:
# Split temporal 80/20 por data da votação (sem vazamento)
df = df.sort_values('data').reset_index(drop=True)
data_corte = df['data'].quantile(0.8)

treino = df[df['data'] < data_corte].copy()
teste = df[df['data'] >= data_corte].copy()

print(f'data_corte = {data_corte.date()}')
print(f'n_treino = {len(treino):>6}  ({treino["data"].min().date()} → {treino["data"].max().date()})')
print(f'n_teste  = {len(teste):>6}  ({teste["data"].min().date()} → {teste["data"].max().date()})')
print('\nClasses no treino:', treino['voto'].value_counts().to_dict())
print('Classes no teste :', teste['voto'].value_counts().to_dict())

assert treino['alvo'].nunique() == 2 and teste['alvo'].nunique() == 2, \
    'Treino ou teste sem as duas classes — ajustar o corte temporal.'

data_corte = 2026-04-29
n_treino =  44043  (2025-02-11 → 2026-04-28)
n_teste  =  11190  (2026-04-29 → 2026-06-03)

Classes no treino: {'favoravel': 27415, 'contrario': 16628}
Classes no teste : {'favoravel': 7791, 'contrario': 3399}


## 3. Feature engineering

Features categóricas (`partido`, `uf`, `tema_cluster`, `mes`, **`bloco`**) via one-hot +
uma feature numérica: `tend_partido_tema` (proporção histórica de votos favoráveis daquele
partido naquele tema). **Crítico**: essa tendência é calculada **somente no treino** — usá-la
sobre todo o dataset seria vazamento temporal.

A feature **`bloco`** (coalizão) resume cada partido em `governo` / `oposicao` / `centro`
no governo Lula (57ª legislatura). A literatura de ciência política aponta o alinhamento
governo×oposição como o maior preditor de voto no Congresso brasileiro — é o sinal que
faltava ao modelo v1, que só tinha o partido pulverizado no one-hot.

In [4]:
# Feature derivada SÓ do treino: tendência do partido naquele tema (evita vazamento).
media_global = treino['alvo'].mean()
tend = (
    treino.groupby(['partido', 'tema_cluster'])['alvo']
    .mean()
    .rename('tend_partido_tema')
    .reset_index()
)

# Feature de COALIZÃO: bloco do partido no governo Lula (57ª legislatura, 2023–2026).
# Enquadramento político público; partidos não listados caem em 'centro' (centrão,
# voto variável). É a variável que a literatura de ciência política aponta como o
# maior preditor de voto no Congresso brasileiro — disciplina partidária por bloco.
BLOCO_PARTIDO = {
    # base governista
    'PT': 'governo', 'PSB': 'governo', 'PCDOB': 'governo', 'PV': 'governo',
    'PSOL': 'governo', 'PDT': 'governo', 'REDE': 'governo',
    'SOLIDARIEDADE': 'governo', 'AVANTE': 'governo', 'PROS': 'governo',
    # oposição
    'PL': 'oposicao', 'NOVO': 'oposicao', 'PRTB': 'oposicao',
    # centro / centrão (voto variável, fisiológico)
    'MDB': 'centro', 'PSD': 'centro', 'UNIAO': 'centro', 'REPUBLICANOS': 'centro',
    'PP': 'centro', 'PODE': 'centro', 'PSDB': 'centro', 'CIDADANIA': 'centro',
    'PSC': 'centro', 'PATRIOTA': 'centro', 'PMN': 'centro', 'AGIR': 'centro',
    'PRD': 'centro',
}

def bloco_de(partido):
    chave = str(partido).strip().upper().replace('Ã', 'A')
    if chave.startswith('UNIAO'):  # 'UNIÃO' / 'UNIÃO BRASIL'
        chave = 'UNIAO'
    return BLOCO_PARTIDO.get(chave, 'centro')

def add_features(d):
    d = d.merge(tend, on=['partido', 'tema_cluster'], how='left')
    # combinações partido×tema inéditas no teste recebem a média global do treino
    d['tend_partido_tema'] = d['tend_partido_tema'].fillna(media_global)
    d['mes'] = d['data'].dt.month
    d['bloco'] = d['partido'].map(bloco_de)
    return d

treino_f = add_features(treino)
teste_f = add_features(teste)

# 'bloco' entra como categórica (one-hot) junto das demais; resume o partido em 3
# blocos robustos, complementando o one-hot esparso de cada sigla.
COLS_CAT = ['partido', 'uf', 'tema_cluster', 'mes', 'bloco']
COLS_NUM = ['tend_partido_tema']

def montar_X(d):
    return pd.get_dummies(d[COLS_CAT + COLS_NUM], columns=COLS_CAT)

X_treino = montar_X(treino_f)
# Realinha o teste às colunas do treino (categorias ausentes viram 0; evita vazamento de schema)
X_teste = montar_X(teste_f).reindex(columns=X_treino.columns, fill_value=0)
y_treino = treino_f['alvo'].values
y_teste = teste_f['alvo'].values

print(f'X_treino: {X_treino.shape}   X_teste: {X_teste.shape}')
print('\nDistribuição de bloco (treino):')
print(treino_f['bloco'].value_counts())
print('\n% favorável por bloco (treino) — sanidade da feature de coalizão:')
print(treino_f.groupby('bloco')['alvo'].mean().round(3))

X_treino: (44043, 63)   X_teste: (11190, 63)

Distribuição de bloco (treino):
bloco
centro      22651
governo     12578
oposicao     8814
Name: count, dtype: int64

% favorável por bloco (treino) — sanidade da feature de coalizão:
bloco
centro      0.635
governo     0.635
oposicao    0.571
Name: alvo, dtype: float64


## 4. Treino, busca de hiperparâmetros e threshold

Três melhorias sobre o modelo v1:

1. **Busca de hiperparâmetros** (`RandomizedSearchCV`) com **`TimeSeriesSplit`** — a
   validação cruzada respeita a ordem temporal (treina no passado, valida no futuro),
   sem vazamento. Otimiza `f1_macro`.
2. **Análise de threshold** numa validação temporal (cauda do treino). O corte que
   maximiza F1 na validação **não generaliza** — a taxa-base de "favorável" varia entre
   períodos — então adotamos o padrão **0,50**, que maximiza a acurácia (métrica da
   hipótese) e é o ponto de operação mais robusto. A análise fica documentada como lição.
3. **Feature de coalizão** (`bloco`) já incorporada na etapa 3.

`class_weight='balanced'` segue compensando o desbalanceamento (favorável domina).

In [5]:
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit
from sklearn.base import clone
from scipy.stats import randint

# --- 4a. Busca de hiperparâmetros (TimeSeriesSplit — sem vazamento temporal) ---
tscv = TimeSeriesSplit(n_splits=3)
espaco = {
    'n_estimators': randint(200, 500),
    'max_depth': [None, 12, 20, 30],
    'min_samples_leaf': randint(1, 20),
    'max_features': ['sqrt', 'log2', 0.3],
}
busca = RandomizedSearchCV(
    RandomForestClassifier(random_state=42, class_weight='balanced', n_jobs=1),
    espaco, n_iter=15, scoring='f1_macro', cv=tscv,
    random_state=42, n_jobs=-1, verbose=0,
)
busca.fit(X_treino, y_treino)
modelo = busca.best_estimator_  # já reajustado em todo o treino (refit=True)
modelo.n_jobs = -1
print('Melhores hiperparâmetros:', busca.best_params_)
print(f'F1 macro (CV temporal no treino): {busca.best_score_:.4f}')

# --- 4b. Análise de threshold numa fatia de VALIDAÇÃO temporal (cauda do treino) ---
# O treino está ordenado por data; separamos os 20% finais como validação. Um clone é
# treinado só no passado e medimos o threshold que maximiza F1_macro na validação.
n_val = int(len(X_treino) * 0.2)
X_tr_in, X_val = X_treino.iloc[:-n_val], X_treino.iloc[-n_val:]
y_tr_in, y_val = y_treino[:-n_val], y_treino[-n_val:]

m_thr = clone(modelo).fit(X_tr_in, y_tr_in)
proba_val = m_thr.predict_proba(X_val)[:, 1]
grade_thr = np.arange(0.25, 0.71, 0.01)
f1s = [f1_score(y_val, (proba_val >= t).astype(int), average='macro') for t in grade_thr]
thr_val = float(grade_thr[int(np.argmax(f1s))])
print(f'Threshold que maximiza F1 na validação: {thr_val:.2f}  |  F1_macro val: {max(f1s):.4f}')

# --- 4c. Avaliação no teste em dois pontos de operação ---
proba_teste = modelo.predict_proba(X_teste)[:, 1]

def avaliar(thr):
    p = (proba_teste >= thr).astype(int)
    return accuracy_score(y_teste, p), f1_score(y_teste, p, average='macro'), p

acc_05, f1_05, pred_05 = avaliar(0.50)
acc_val, f1_val, _ = avaliar(thr_val)

print(f'\n--- thr=0.50 (padrão):          acuracia={acc_05:.4f}  F1_macro={f1_05:.4f}')
print(f'--- thr={thr_val:.2f} (ótimo na val):   acuracia={acc_val:.4f}  F1_macro={f1_val:.4f}')

# O threshold ótimo na validação NÃO generaliza: a taxa-base de "favorável" muda entre
# períodos (a validação tinha mais votos contrários que o teste recente). Como a hipótese
# do projeto é medida em ACURÁCIA, adotamos o ponto de operação padrão (0,50) — o mais
# robusto e de maior acurácia no teste.
thr_final = 0.50
acuracia, f1m, pred = acc_05, f1_05, pred_05
print(f'\n>>> Ponto de operação adotado: thr={thr_final:.2f}  (acuracia={acuracia:.4f}, F1_macro={f1m:.4f})')
meta = 'ATINGIDA ✅' if acuracia >= 0.70 else 'NÃO atingida ❌'
print(f'Meta da hipótese (acurácia >= 0.70): {meta}')

print('\nMatriz de confusão (linhas=real, colunas=previsto) [0=contrario, 1=favoravel]:')
print(confusion_matrix(y_teste, pred))

print('\n' + classification_report(y_teste, pred, target_names=['contrario', 'favoravel']))

imp = pd.Series(modelo.feature_importances_, index=X_treino.columns).sort_values(ascending=False)
print('Top-15 features mais importantes:')
print(imp.head(15).to_string())

Melhores hiperparâmetros: {'max_depth': 30, 'max_features': 0.3, 'min_samples_leaf': 3, 'n_estimators': 334}
F1 macro (CV temporal no treino): 0.5312


Threshold que maximiza F1 na validação: 0.58  |  F1_macro val: 0.4976



--- thr=0.50 (padrão):          acuracia=0.6256  F1_macro=0.5567
--- thr=0.58 (ótimo na val):   acuracia=0.3828  F1_macro=0.3733

>>> Ponto de operação adotado: thr=0.50  (acuracia=0.6256, F1_macro=0.5567)
Meta da hipótese (acurácia >= 0.70): NÃO atingida ❌

Matriz de confusão (linhas=real, colunas=previsto) [0=contrario, 1=favoravel]:
[[1295 2104]
 [2086 5705]]

              precision    recall  f1-score   support

   contrario       0.38      0.38      0.38      3399
   favoravel       0.73      0.73      0.73      7791

    accuracy                           0.63     11190
   macro avg       0.56      0.56      0.56     11190
weighted avg       0.62      0.63      0.63     11190

Top-15 features mais importantes:
tend_partido_tema    0.283194
mes_10               0.060793
tema_cluster_8       0.041371
mes_3                0.035994
tema_cluster_3       0.035821
tema_cluster_1       0.034198
bloco_oposicao       0.030823
mes_2                0.025829
partido_PL           0.022605
me

## 5. Gravar métricas no Supabase

Registra o resultado do treino em `metricas_modelo` (append-only — mantém o histórico de runs).

In [6]:
registro = {
    'modelo': 'random_forest_v2_coalizao',
    'acuracia': round(float(acuracia), 4),
    'f1_macro': round(float(f1m), 4),
    'data_corte': str(data_corte.date()),
    'n_treino': int(len(treino)),
    'n_teste': int(len(teste)),
}
inserir_metricas(registro)
print('Métricas (v2) gravadas em metricas_modelo:')
for k, v in registro.items():
    print(f'  {k}: {v}')
print(f'  ponto de operação: thr={thr_final:.2f}')
print(f'  hiperparâmetros: {busca.best_params_}')

2026-06-25 23:58:58,502 [INFO] HTTP Request: POST https://woxluznsfqskopzzjqtf.supabase.co/rest/v1/metricas_modelo "HTTP/2 201 Created"


2026-06-25 23:58:58,505 [INFO] inserir metricas_modelo: random_forest_v2_coalizao


Métricas (v2) gravadas em metricas_modelo:
  modelo: random_forest_v2_coalizao
  acuracia: 0.6256
  f1_macro: 0.5567
  data_corte: 2026-04-29
  n_treino: 44043
  n_teste: 11190
  ponto de operação: thr=0.50
  hiperparâmetros: {'max_depth': 30, 'max_features': 0.3, 'min_samples_leaf': 3, 'n_estimators': 334}
